In [5]:
#load libraries
require(tidyverse)
require(dplyr)

Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


# master datasheet including:
- sampling info
- colony conditions
- post dna extraction info

In [17]:
#load Colony_Data.csv
colony=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/CBC_ColonyData.csv")

In [19]:
#cleaning it up to only include conditions, transect num and tag num
#cclean to just include Condition columns and colony'
colony=colony[c('TransectNum', 'NewTagNum', 'X062019_Condition', 'X102019_Condition', 'X052022_Condition', 'X122022_Condition', 'X092023_Condition', 'X112023_Condition', 'X122023_Condition','X012024_Condition','X022024_Condition', 'X042024_Condition', 'X062024_Condition', 'X082024_Condition', 'immune_y.n')]
head(colony)

,TransectNum,NewTagNum,X062019_Condition,X102019_Condition,X052022_Condition,X122022_Condition,X092023_Condition,X112023_Condition,X122023_Condition,X012024_Condition,X022024_Condition,X042024_Condition,X062024_Condition,X082024_Condition,immune_y.n
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,1,Healthy,,Diseased,Diseased,Diseased,Not_Visited,Not_Visited,Diseased,Not_Visited,Diseased,Not_Visited,Not_Visited,n
2,1,2,Healthy,,Healthy,Healthy,"CLP,CLB",CLB,CLB,"CLP,CLB",Healthy,Not_Visited,Healthy,Healthy,y
3,1,3,Healthy,,Diseased,Healthy,Diseased,CLP,Healthy,Healthy,Diseased,Diseased,Diseased,Diseased,y
4,1,4,Healthy,Healthy,Diseased,Dead,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,n
5,1,5,Healthy,,Diseased,Diseased,Diseased,Not_Visited,Not_Visited,Diseased,Not_Visited,Diseased,Not_Visited,Not_Visited,n
6,1,6,Healthy,Healthy,Healthy,Diseased,Dead,Not_Visited,Not_Visited,Dead,Not_Visited,Not_Visited,Not_Visited,Not_Visited,n


In [20]:
#isolate immune samples from colony data
colony_immune=colony[colony$immune_y.n=="y",]
# new character colony to use for samples
colony_character=as.vector(paste0(colony_immune$TransectNum,"_", colony_immune$NewTagNum))
#make new column colony in colony data to match meta
colony_immune$colony <- c(paste0(colony_immune$TransectNum, "_", colony_immune$NewTagNum))

In [21]:
#rename condition columns to corresponding Monthyear
colony_clean <- colony_immune %>%
  rename_with(
    .cols = matches("^X\\d{6}_Condition$"),
    .fn = ~ str_extract(., "\\d{6}")
  )

#pivot longer colony to join all date columns into one monthyear column
colony_long <- colony_clean %>%
  pivot_longer(
    cols = matches("^\\d{6}$"),      # after renaming
    names_to = "Month_year",
    values_to = "Condition"
  )
head(colony_long)

#remove now unnecessary transect num and tag num
colony_long[ ,c('TransectNum', 'NewTagNum')] <- list(NULL)
head(colony_long)

TransectNum,NewTagNum,immune_y.n,colony,Month_year,Condition
<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,2,y,1_2,062019,Healthy
1,2,y,1_2,102019,
1,2,y,1_2,052022,Healthy
1,2,y,1_2,122022,Healthy
1,2,y,1_2,092023,"CLP,CLB"
1,2,y,1_2,112023,CLB


immune_y.n,colony,Month_year,Condition
<chr>,<chr>,<chr>,<chr>
y,1_2,062019,Healthy
y,1_2,102019,
y,1_2,052022,Healthy
y,1_2,122022,Healthy
y,1_2,092023,"CLP,CLB"
y,1_2,112023,CLB


In [22]:
class(colony_immune)
class(colony_long)
class(colony_character)

[1] "data.frame"

[1] "tbl_df"     "tbl"        "data.frame"

[1] "character"

In [23]:
#load metagenomics data
metagenomics=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/Metagenomics_Tracker_Belize.csv")

In [24]:
#using an old copy of sample data from github bc i messed around with it and need to wait for pull review
sample=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/CBC_samples.csv")

# sample_dna add colony column to sample data
sample$colony= colony_character= c(paste0(sample$TransectNum, "_", sample$NewTagNum))

#leading zeros are gone, add them back in
sample$Month_year <- sprintf("%06d", as.numeric(sample$Month_year))

# only utilize EtOH and RNALater samples and exclue 062025 samples
sample_DNA <- sample[(sample$Sample_type == "Core_EtOH" | sample$Sample_type == "Core_RNAlater") &
    sample$Month_year != "062025",]

#double check that all 062025 samples have been removed
unique(sample_DNA$Month_year)


#view sample_dna, this is a new table modeled after sample data sheet and utilizing the colony tag num as a column
head(sample_DNA)



[1] "092023" "122022" "052022" "112023" "042024" "102019" "062019" "122023"
 [9] "012024" "022024" "062024" "082024"

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,⋯,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,092023,BEL,CBC,9/27/23,CBC30N,1,,1,SSID,,⋯,Core_RNAlater,185,Diseased_Margin,only margin sample available,092023_BEL_CBC_T1_185_SSID,UML_NARWHAL_R1_B10,,,,1_1
3,092023,BEL,CBC,9/25/23,CBC30N,1,,2,PAST,,⋯,Core_RNAlater,171,Bleached_Tissue,CLP 90%,092023_BEL_CBC_T1_171_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_2
4,092023,BEL,CBC,9/25/23,CBC30N,1,,3,SSID,,⋯,Core_RNAlater,173,Healthy,CLP 80%; DC 20%,092023_BEL_CBC_T1_173_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_3
5,092023,BEL,CBC,9/25/23,CBC30N,1,,12,PSTR,,⋯,Core_RNAlater,177,Healthy,No CL,092023_BEL_CBC_T1_177_PSTR,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_12
6,092023,BEL,CBC,9/25/23,CBC30N,1,,13,PAST,,⋯,Core_RNAlater,175,Healthy,No CL,092023_BEL_CBC_T1_175_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,R2_B15 EXTRACTED TWICE,1_13
7,092023,BEL,CBC,9/27/23,CBC30N,1,,5,SSID,,⋯,Core_RNAlater,188,Diseased_Tissue,,092023_BEL_CBC_T1_188_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B11,,,1_5


In [25]:
## now combine sample and colony data by month_year and colony
#merge new colony_long and meta
meta<- merge(sample_DNA, colony_long,
                     by = c("Month_year", "colony"),
                     all = FALSE)

In [26]:
head(meta)
nrow(meta)

,Month_year,colony,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,⋯,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,immune_y.n,Condition
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,012024,1_12,BEL,CBC,1/10/24,CBC30N,1,,12,PSTR,⋯,563,Healthy,,012024_BEL_CBC_T1_563_PSTR,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
2,012024,1_21,BEL,CBC,1/10/24,CBC30N,1,,21,PAST,⋯,565,Bleached_Tissue,,012024_BEL_CBC_T1_565_PAST,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,CLP
3,012024,1_24,BEL,CBC,1/10/24,CBC30N,1,,24,MCAV,⋯,559,Healthy,,012024_BEL_CBC_T1_559_MCAV,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
4,012024,1_25,BEL,CBC,1/10/24,CBC30N,1,,25,OANN,⋯,561,Healthy,,012024_BEL_CBC_T1_561_OANN,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
5,012024,1_3,BEL,CBC,1/10/24,CBC30N,1,,3,SSID,⋯,557,Healthy,,012024_BEL_CBC_T1_557_SSID,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
6,012024,2_55,BEL,CBC,1/12/24,SR30N,2,,55,MCAV,⋯,603,Bleached_Tissue,Healthy,012024_BEL_CBC_T2_603_MCAV,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,CLP


[1] 369

In [27]:
#changing X column to Tubelabel_species
names(metagenomics)[names(metagenomics) == 'X'] <- 'Tubelabel_species'

In [28]:
colnames(metagenomics)

[1] "Tubelabel_species"                                                  
 [2] "Health_Status"                                                      
 [3] "Starting_Weight"                                                    
 [4] "Date_Extracted"                                                     
 [5] "Raw_ng_ul"                                                          
 [6] "Date_Enriched"                                                      
 [7] "Microbe_ng_ul"                                                      
 [8] "Microbe_Location"                                                   
 [9] "Microbe_clean_date.n"                                               
[10] "Host_ng_ul"                                                         
[11] "Host_Location"                                                      
[12] "Host_clean_y.n"                                                     
[13] "Date_Libprep"                                                       
[14] "Status"                                                             
[15] "Notes"                                                              
[16] "Seq_date"                                                           
[17] "Host_Seq_date"                                                      
[18] "Microbe_seq_file"                                                   
[19] "Host_seq_file"                                                      
[20] "Seq_Location..in.Unity."                                            
[21] "Sample_physical_location"                                           
[22] "Extraction_physical_location"                                       
[23] "Location_notes"                                                     
[24] "Sample_Code...datecollected_tagnumber_transect_samplenumber_species"

In [29]:
#the columns that I want to isolate from metagenomics tracker
# not grabbing Health_status from metagenomics bc I want to get it from CBC_samples
metagenomics_PCR = metagenomics[c("Tubelabel_species", "Date_Extracted", "Raw_ng_ul", "Date_Enriched","Microbe_Location")]

In [30]:
head(metagenomics_PCR)

,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location
,<chr>,<chr>,<chr>,<chr>,<chr>
1,052022_BEL_CBC_T2_45_PSTR,10_10_2023,22.8,10_17_2023,UML_NARWHAL_R2_B1
2,052022_BEL_CBC_T2_46_PSTR,10_10_2023,63.9,10_17_2023,UML_NARWHAL_R2_B1
3,052022_BEL_CBC_T2_59_OFAV,10_3_2023,73.7,10_16_2023,UML_NARWHAL_R2_B1
4,042024_BEL_CBC_T1_925_PAST,8_19_2024,5.46,9_28_2024,UML_NARWHAL_R6_B30
5,052022_BEL_CBC_T2_72_OFAV,10_3_2023,47.6,10_11_23,UML_NARWHAL_R6_B1
6,052022_BEL_CBC_T2_71_OFAV,10_3_2023,98.8,10_16_2023,UML_NARWHAL_R6_B1


In [31]:
nrow(metagenomics_PCR)

[1] 558

In [32]:
nrow(meta)

[1] 369

In [34]:
#check to see why there is a discrepancy from immune_sample_DNA to_metagenomics_PCR
unmatched=anti_join(meta,metagenomics_PCR, by='Tubelabel_species')
nrow(unmatched)
unique(unmatched$Tubelabel_species)

[1] 0

character(0)

In [35]:
# inner join 
immune_meta_PCR <- merge(x=metagenomics_PCR,y=meta, 
                by="Tubelabel_species")
head(immune_meta_PCR)

,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,Country,Location,CollectionDate,⋯,Sample_type,SampleNum,Health_status,Sampling_notes,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,immune_y.n,Condition
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,012024_BEL_CBC_T1_557_SSID,2_17_2025,44.6,,,012024,1_3,BEL,CBC,1/10/24,⋯,Core_RNAlater,557,Healthy,,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
2,012024_BEL_CBC_T1_559_MCAV,2_17_2025,14.8,,,012024,1_24,BEL,CBC,1/10/24,⋯,Core_RNAlater,559,Healthy,,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
3,012024_BEL_CBC_T1_561_OANN,2_17_2025,18,,,012024,1_25,BEL,CBC,1/10/24,⋯,Core_RNAlater,561,Healthy,,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
4,012024_BEL_CBC_T1_563_PSTR,2_17_2025,35.8,,,012024,1_12,BEL,CBC,1/10/24,⋯,Core_RNAlater,563,Healthy,,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy
5,012024_BEL_CBC_T1_565_PAST,2_17_2025,7.72,,,012024,1_21,BEL,CBC,1/10/24,⋯,Core_RNAlater,565,Bleached_Tissue,,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,CLP
6,012024_BEL_CBC_T2_585_OFAV,2_17_2025,71.6,,,012024,2_76,BEL,CBC,1/12/24,⋯,Core_RNAlater,585,Healthy,Healthy,UML_NARWHAL_R5_B18,UML_NARWHAL_R2_B16,,,y,Healthy


In [36]:
colnames(immune_meta_PCR)

[1] "Tubelabel_species"            "Date_Extracted"              
 [3] "Raw_ng_ul"                    "Date_Enriched"               
 [5] "Microbe_Location"             "Month_year"                  
 [7] "colony"                       "Country"                     
 [9] "Location"                     "CollectionDate"              
[11] "Transect"                     "TransectNum"                 
[13] "OldTagNum"                    "NewTagNum"                   
[15] "Species"                      "Time_sampled"                
[17] "Time_processed"               "Sample_type"                 
[19] "SampleNum"                    "Health_status"               
[21] "Sampling_notes"               "Sample_physical_location"    
[23] "Extraction_physical_location" "Date_sequenced"              
[25] "Notes"                        "immune_y.n"                  
[27] "Condition"

In [37]:
## columns I dont need in immune_meta_pcr
immune_meta_PCR[ ,c('Country', 'Location', 'Transect', 'TransectNum', 'OldTagNum', 'NewTagNum', 'Time_sampled', 'Time_processed', 'Sample_type', 'SampleNum', 'Sampling_notes', 'Date_sequenced', 'Notes')] <- list(NULL)

In [38]:
length(unique(immune_meta_PCR$colony))

[1] 37

In [39]:
nrow(meta)

[1] 369

In [40]:
nrow(metagenomics_PCR)

[1] 558

In [153]:
# manually corrected for these unmatched values in metagenomics tracker and CBC_samples
# mostly my own typos
# find 052022_BEL_CBC_T3_25_PAST extraction
# do we want to extract 012024?

In [42]:
#make spreadsheet with all immune metagenomics 
write.csv(immune_meta_PCR, file="~/Documents/SCTLD/Coral_Code/NSF_Rapid/CBC_PCR_Plan/immune_PCR.csv")

# combining the immune and non_immune datasheets 
- coalesing them into the same order as my separate randomized datasheet

In [43]:
# Step 1: Load your data
immune <- read.csv("immune_PCR.csv")

nonimmune <- read.csv("2_nonimmune_PCR.csv")

#combine these dataframes
meta <- rbind(immune, nonimmune)

In [44]:
#now I want to coelese the order so the order of the samples matched 3_randomized_nonimmune_metagenomics_PCR
#first I'll make "Date_16S" and Date_ITS2 columns in new sheet

old=read.csv("~/Documents/SCTLD/Coral_Code/NSF_Rapid/CBC_PCR_Plan/3_randomized_nonimmune_PCR.csv")
nrow(meta)
nrow(old)

[1] 493

[1] 471

In [45]:
# Step 1: Create 'Date_16S' column in df 'two' with NA values
meta <- meta %>%
  mutate(Date_16S = NA)

# Step 2: Coalesce values (e.g., combine one$Date_16S and two$Date_16S)
meta$Date_16S <- coalesce(old$Date_16S[match(meta$Tubelabel_species, old$Tubelabel_species)], meta$Date_16S)

meta <- meta %>%
  mutate(
    # Ensure character
    Date_16S = as.character(Date_16S),

    # Replace blanks with NA
    Date_16S = ifelse(Date_16S == "", NA, Date_16S),

    # Temporarily convert to Date object
    Date_parsed = as.Date(Date_16S, format = "%m_%d_%Y")
  ) %>%
  
  # Sort using parsed Date
  arrange(is.na(Date_parsed), Date_parsed) %>%
  
  # Restore original format (m_d_Y) for display
  mutate(
    Date_16S = ifelse(
      is.na(Date_parsed),
      NA,
      format(Date_parsed, "%-m_%-d_%Y")  # e.g., 1_2_2025
    )
  ) %>%
  
  # Drop the temporary parsed date column
  select(-Date_parsed)

# make sure the above code did what I asked
head(meta)
unique(meta$Date_16S)

,X,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,Sample_physical_location,Extraction_physical_location,immune_y.n,Condition,Date_16S
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,48,022024_BEL_CBC_T2_814_MCAV,11_22_2024,58.2,,,22024,2_60,2/22/24,MCAV,Healthy,UML_NARWHAL_R5_B23,UML_NARWHAL_R2_B13,y,Healthy,1_2_2025
2,85,042024_BEL_CBC_T2_1144_MCAV,12_2_2024,75.1,,,42024,2_69,4/30/24,MCAV,Healthy,UML_NARWHAL_R5_B25,UML_NARWHAL_R2_B14,y,Healthy,1_2_2025
3,96,042024_BEL_CBC_T3_960_OFAV,12_3_2024,31.8,,,42024,3_66,4/25/24,OFAV,Healthy,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B14,y,Healthy,1_2_2025
4,123,052022_BEL_CBC_T3_13_MCAV,6_15_2022,3,10_30_24,UML_NARWHAL_R6_B27,52022,3_14,5/20/22,MCAV,Healthy,UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B35,y,Healthy,1_2_2025
5,126,052022_BEL_CBC_T3_19_MCAV,2022,17.4,10_30_24,UML_NARWHAL_R6_B27,52022,3_21,5/20/22,MCAV,Healthy,Depleted_UML_NARWHAL_R1_B3,UML_NARWHAL_R2_B32,y,Healthy,1_2_2025
6,181,062024_BEL_CBC_T3_1433_PSTR,12_5_2024,18.6,,,62024,3_70,6/20/24,PSTR,Healthy,UML_NARWHAL_R5_B26,UML_NARWHAL_R2_B15,y,Healthy,1_2_2025


[1] "1_2_2025"   "1_22_2025"  "1_23_2025"  "2_24_2025"  "2_25_2025" 
 [6] "2_26_2025"  "2_27_2025"  "3_4_2025"   "3_6_2025"   "3_7_2025"  
[11] "3_10_2025"  "8_8_2025"   "8_11_2025"  "8_22_2025"  "8_26_2025" 
[16] "8_28_2025"  "9_10_2025"  "10_14_2025" "12_3_2025"  "12_4_2025" 
[21] "1_7_2026"   "1_8_2026"   "1_9_2026"   "2_9_2026"   "2_10_2026" 
[26] "2_11_2026"  "3_9_2026"   "3_10_2026"  "3_11_2026"  NA

In [46]:
#repeat with ITS2
# Step 1: Create 'Date_ITS2' column in df 'two' with NA values
meta <- meta %>%
  mutate(Date_ITS2 = NA)

# Step 2: Coalesce values (e.g., combine one$Date_16S and two$Date_16S)
meta$Date_ITS2 <- coalesce(old$Date_ITS2[match(meta$Tubelabel_species, old$Tubelabel_species)], meta$Date_ITS2)

meta <- meta %>%
  mutate(
    # Ensure character
    Date_ITS2 = as.character(Date_ITS2),

    # Replace blanks with NA
    Date_ITS2 = ifelse(Date_ITS2 == "", NA, Date_ITS2),

    # Temporarily convert to Date object
    Date_ITS2_parsed = as.Date(Date_ITS2, format = "%m_%d_%Y")
  ) %>%
  
  # Sort using parsed Date
  arrange(is.na(Date_ITS2_parsed), Date_ITS2_parsed) %>%
  
  # Restore original format (m_d_Y) for display
  mutate(
    Date_ITS2 = ifelse(
      is.na(Date_ITS2_parsed),
      NA,
      format(Date_ITS2_parsed, "%-m_%-d_%Y")  # e.g., 1_2_2025
    )
  ) %>%
  
  # Drop the temporary parsed date column
  select(-Date_ITS2_parsed)

#make sure it worked
head(meta)
unique(meta$Date_ITS2)

,X,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,Sample_physical_location,Extraction_physical_location,immune_y.n,Condition,Date_16S,Date_ITS2
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,104,052022_BEL_CBC_T1_2_SSID,12_2_2024,7.73,,,52022,1_3,5/21/22,SSID,Diseased_Tissue,UML_NARWHAL_R1_B3,"UML_NARWHAL_R2_B14, UML_NARWHAL_R2_B33",y,Diseased,1_22_2025,6_26_2025
2,231,092023_BEL_CBC_T1_175_PAST,11_8_2024,0.14,,,92023,1_13,9/25/23,PAST,Healthy,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,y,Healthy,1_22_2025,6_26_2025
3,232,092023_BEL_CBC_T1_175_PAST,1_15_2025,2.3,,,92023,1_13,9/25/23,PAST,Healthy,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,y,Healthy,1_22_2025,6_26_2025
4,59,022024_BEL_CBC_T3_851_PSTR,11_26_2024,64.2,,,22024,3_70,2/24/24,PSTR,Healthy,UML_NARWHAL_R5_B23,UML_NARWHAL_R2_B13,y,Healthy,1_23_2025,6_26_2025
5,225,082024_BEL_CBC_T4_1609_OFAV,1_15_2025,61.2,,,82024,4_78,8/22/24,OFAV,Healthy,UML_NARWHAL_R5_B27,UML_NARWHAL_R2_B15,y,Healthy,1_23_2025,6_26_2025
6,292,112023_BEL_CBC_T3_355_PSTR,11_15_2024,1.27,,,112023,3_70,11/13/23,PSTR,Bleached_Tissue,UML_NARWHAL_R5_B19,UML_NARWHAL_R2_B12,y,"CLP,CLB",1_23_2025,6_26_2025


[1] "6_26_2025"  "6_27_2025"  "6_30_2025"  "7_2_2025"   "7_3_2025"  
 [6] "7_7_2025"   "7_9_2025"   "7_10_2025"  "7_11_2025"  "12_10_2025"
[11] "12_15_2025" "12_16_2025" "12_17_2025" "12_18_2025" "12_19_2025"
[16] "12_22_2025" "1_19_2026"  "1_20_2026"  "1_21_2026"  "2_12_2026" 
[21] "2_17_2026"  "2_23_2026"  NA

### why are samples from old being removed from meta
because i put extra filtering perimeters to remove non-immune colonies that were dead before 052022

### okay now without filtering samples dead before 052022
#### why did all of the 9 102019 and 1 042024 PAST get taken out??
because there was no 102019 conditions column in colony

- and 042024_929_PAST was recently updated as an accidental sample
- kick 042024_BEL_CBC_T1_929_PAST out of my dataset

In [47]:
any(is.na(match(old$Tubelabel_species, meta$Tubelabel_species))) #If TRUE, some IDs in old don’t exist in meta.
#check to see why there is a discrepancy from old to meta
inold=anti_join(old,meta, by='Tubelabel_species')
nrow(inold)
unique(inold$Tubelabel_species)

[1] TRUE

[1] 1

[1] "042024_BEL_CBC_T1_929_PAST"

In [48]:
any(is.na(match(meta$Tubelabel_species, old$Tubelabel_species)))
#check to see why there is a discrepancy from meta to old
inmeta=anti_join(meta,old, by='Tubelabel_species')
nrow(inmeta)
unique(inmeta$Tubelabel_species)

[1] TRUE

[1] 14

[1] "022024_BEL_CBC_T3_846_OFAV" "122022_BEL_CBC_T3_147_OFAV"
 [3] "062019_BEL_CBC_T1_4_MCAV"   "062019_BEL_CBC_T2_28_MCAV" 
 [5] "062019_BEL_CBC_T3_28_SSID"  "102019_BEL_CBC_T1_26_PSTR" 
 [7] "102019_BEL_CBC_T1_30_PSTR"  "102019_BEL_CBC_T2_36_PSTR" 
 [9] "102019_BEL_CBC_T3_34_PSTR"  "102019_BEL_CBC_T3_37_PSTR" 
[11] "102019_BEL_CBC_T3_39_PSTR"  "122022_BEL_CBC_T3_145_MCAV"
[13] "122022_BEL_CBC_T3_158_PSTR"

#### why are some samples in meta but not in old datasheet?
because they died before 052022

In [49]:
# make sure 3 randomized and meta pcr are in the same order bc of indexing set up

# Get match positions
ord <- match(meta$Tubelabel_species, old$Tubelabel_species)

# Reorder: matched rows first (in old's order), unmatched at bottom
meta <- meta[order(is.na(ord), ord), ]
head(meta)

,X,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,CollectionDate,Species,Health_status,Sample_physical_location,Extraction_physical_location,immune_y.n,Condition,Date_16S,Date_ITS2
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,104,052022_BEL_CBC_T1_2_SSID,12_2_2024,7.73,,,52022,1_3,5/21/22,SSID,Diseased_Tissue,UML_NARWHAL_R1_B3,"UML_NARWHAL_R2_B14, UML_NARWHAL_R2_B33",y,Diseased,1_22_2025,6_26_2025
2,231,092023_BEL_CBC_T1_175_PAST,11_8_2024,0.14,,,92023,1_13,9/25/23,PAST,Healthy,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,y,Healthy,1_22_2025,6_26_2025
3,232,092023_BEL_CBC_T1_175_PAST,1_15_2025,2.3,,,92023,1_13,9/25/23,PAST,Healthy,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,y,Healthy,1_22_2025,6_26_2025
4,59,022024_BEL_CBC_T3_851_PSTR,11_26_2024,64.2,,,22024,3_70,2/24/24,PSTR,Healthy,UML_NARWHAL_R5_B23,UML_NARWHAL_R2_B13,y,Healthy,1_23_2025,6_26_2025
5,225,082024_BEL_CBC_T4_1609_OFAV,1_15_2025,61.2,,,82024,4_78,8/22/24,OFAV,Healthy,UML_NARWHAL_R5_B27,UML_NARWHAL_R2_B15,y,Healthy,1_23_2025,6_26_2025
8,360,122023_BEL_CBC_T3_526_SSID,11_19_2024,16.4,,,122023,3_5,12/16/23,SSID,Healthy,UML_NARWHAL_R5_B19,UML_NARWHAL_R2_B13,y,Healthy,1_23_2025,6_26_2025


In [51]:
# adding leading zeros to Month_year column

meta$Month_year <- sprintf("%06d", as.numeric(meta$Month_year))

In [52]:
write.csv(meta, file="~/Documents/SCTLD/Coral_Code/NSF_Rapid/CBC_PCR_Plan/meta_PCR.csv")